In [0]:
from src.config import *
from src.api_utils import *
from src.schemas import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
token = authenticate()
headers = {
    'Authorization': f'Bearer {token}'
}

In [0]:
for endpoint in VALID_ENDPOINTS:
    
    skip = 0
    schema = ENDPOINT_SCHEMAS[endpoint]

    while True:
        response = fetch_data(endpoint=endpoint, headers=headers, limit=PAGE_SIZE, skip=skip)
        df = spark\
            .createDataFrame(response[endpoint], schema=schema)\
            .withColumn('ingestion_timestamp', F.current_timestamp())\
            .withColumn('source_system', F.lit(f'{BASE_URL}/{endpoint}'))
        
        table_exists = spark.catalog.tableExists(f'{TABLE_PREFIX}.bronze_{endpoint}')
        if table_exists:
            df.writeTo(f'{TABLE_PREFIX}.bronze_{endpoint}').append()
        else:
            df.writeTo(f'{TABLE_PREFIX}.bronze_{endpoint}').create()
        
        skip += response['limit']
        if skip >= response['total']:
            break